# Student Marks Neural Network
*This Jupyter notebook details the construction of a neural network that predicts the math score of a student based from their demographic profile and reading/writing grades.*

In [ ]:
# Import necessary libraries/modules
import os
import pandas as pandas
import tensorflow as tensorflow
from tensorflow import keras
from tensorflow.keras.wrappers.scikit_learn import KerasRegressor
import numpy as numpy
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plots

## Data Preprocessing

In [ ]:
# Load Data from CSV file
student_data = pandas.read_csv('exams.csv')
student_data.info()

In [ ]:
# Replace space characters with underscore, simplified race/ethnicity name
student_data.columns = student_data.columns.str.replace(' ', '_')
student_data.columns = student_data.columns.str.replace('race/', '')

In [ ]:
# Preprocess categorical data to convert it into numerical values
label_encoder = preprocessing.LabelEncoder()
categorical_columns = ['gender', 'ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course']
for column in categorical_columns:
    student_data[column] = label_encoder.fit_transform(student_data[column])
student_data.info()

In [ ]:
# Convert DataFrame into Numpy Array
student_numpy = student_data.to_numpy()

In [ ]:
# Separating feature variables from target variable
x_data = numpy.concatenate((student_numpy[:,0:5], student_numpy[:,6:8]), axis = 1)
y_data = student_numpy[:, 5]

In [ ]:
# Train Scaler Model and use it for scaling data
scaler_model = StandardScaler().fit(x_data)
x_data = scaler_model.transform(x_data)

In [ ]:
# Split training and testing data
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size = 0.10)

## Creating a Model

In [ ]:
target_output = 1
hidden_layer_function = 'sigmoid'
metric = tensorflow.keras.metrics.RootMeanSquaredError()

def create_model():
    # Instantiate Sequential model
    model = tensorflow.keras.models.Sequential()

    # Add hidden layers
    model.add(keras.layers.Dense(4,
                                 input_shape = (7,),
                                 name = "Hidden_Layer_1",
                                 activation = hidden_layer_function))
    # Add output layer
    model.add(keras.layers.Dense(target_output,
                                 name = "Output_Layer"))

    # Compile the model
    model.compile(loss = 'mean_squared_error',
                  optimizer = 'adam',
                  metrics = [metric])
    return model

## Training and Evaluating Model

In [ ]:
# Constants for Training/Evaluation
VERBOSE = 1
BATCH_SIZE = 16
EPOCHS = 2
VALIDATION_SPLIT = 0.2

regressor_model = KerasRegressor(build_fn=create_model, epochs=EPOCHS, batch_size=BATCH_SIZE)

# Train model with hyperparameters
print("Training Progress:\n-----------------------------")
history = regressor_model.fit(x_train,
                              y_train,
                              batch_size = BATCH_SIZE,
                              epochs = EPOCHS,
                              verbose = VERBOSE,
                              validation_split = VALIDATION_SPLIT)

# Check Accuracy after Training
print("\nAccuracy during Training :\n------------------------------------")
pandas.DataFrame(history.history)['root_mean_squared_error'].plot(figsize = (8,5))
plots.title("Accuracy Over Epochs")
plots.show()